# Mobil Yüz Rekonstrüksiyon — Kaggle Eğitim Notebook'u

**GPU:** Kaggle P100/T4 (haftada 30 saat ücretsiz)  
**Çıktılar:** `/kaggle/working/` — session sonunda zip olarak indir  
**Kalıcı checkpoint:** Kaggle Dataset olarak kaydet (aşağıda açıklanıyor)

---
### Adımlar
1. Kurulum
2. Kodu GitHub'dan çek
3. Config (Kaggle yolları)
4. Veri seti hazırlama
5. Ön işleme
6. Eğitim
7. TFLite export
8. Checkpoint'i kalıcı kaydet

## 1. Kurulum

In [ ]:
import subprocess, sys

!nvidia-smi

!pip install -q \
    insightface \
    onnxruntime-gpu \
    facenet-pytorch \
    pytorch-msssim \
    torchmetrics \
    opencv-python-headless \
    onnx \
    tensorboard \
    ai-edge-torch

print('Kurulum tamamlandı.')

## 2. Proje Kodunu GitHub'dan Çek

In [ ]:
import os, sys

!git clone https://github.com/BuhraOzdemir/face-recon.git /kaggle/working/face-recon
sys.path.insert(0, '/kaggle/working/face-recon')
%cd /kaggle/working/face-recon

print('Proje kodu hazır:', os.getcwd())

## 3. Config (Kaggle Yolları)

Kaggle'da klasör yapısı:
```
/kaggle/input/DATASET_ADI/   ← eklediğin veri seti
/kaggle/working/             ← tüm çıktılar (checkpoint, export, log)
```

In [ ]:
from src.config import Config, DataConfig, ModelConfig, LossConfig, TrainConfig, ExportConfig

# Kaggle'a eklediğin veri setinin adını buraya yaz
# Notebook'un sag panelinde dataseti gordugunde URL'deki slug kismini al
# Ornek: kaggle.com/datasets/nguyncngtr/ms1mv3 -> 'ms1mv3'
DATASET_NAME = 'ms1mv3'  # <-- ekledığın datasetin Kaggle slug adı

cfg = Config(
    data=DataConfig(
        data_dir      = f'/kaggle/input/{DATASET_NAME}',
        processed_dir = '/kaggle/working/processed',
        image_size    = 128,
        val_split     = 0.05,
        num_workers   = 2,
    ),
    model=ModelConfig(
        embedding_dim    = 512,
        initial_spatial  = 4,
        initial_channels = 128,
        decoder_channels = (128, 128, 64, 32, 16),
    ),
    loss=LossConfig(
        phase1_epochs      = 10,
        phase2_epochs      = 50,
        phase3_epochs      = 40,
        vgg_layer          = 'relu3_3',
        facenet_input_size = 160,
    ),
    train=TrainConfig(
        epochs            = 100,
        batch_size        = 64,
        learning_rate     = 1e-4,
        weight_decay      = 1e-4,
        warmup_epochs     = 5,
        save_dir          = '/kaggle/working/checkpoints',
        save_every_epochs = 5,
        keep_last_n       = 3,
        patience          = 10,
        use_amp           = True,
        log_dir           = '/kaggle/working/logs',
        log_every_steps   = 50,
    ),
    export=ExportConfig(
        export_dir  = '/kaggle/working/export',
        model_name  = 'face_decoder',
        quantize_int8 = True,
    ),
)

print(f'Toplam epoch : {cfg.total_epochs()}')
print(f'Veri giriş   : {cfg.data.data_dir}')
print(f'Checkpoint   : {cfg.train.save_dir}')
print(f'Export       : {cfg.export.export_dir}')

## 4. Veri Seti — Format Tespiti ve Çıkarım

MS1MV3 genellikle **InsightFace RecordIO** formatında gelir (`.rec` + `.idx` + `property`).  
Bu hücre formatı otomatik tespit eder ve RecordIO ise görüntüleri çıkarır.

> **Dataset slug'ını bulmak için:** Kaggle'da sağ panelde dataseti görünce 
> `▼ Input` altındaki yola bak: `/kaggle/input/SLUG_BURADA/`

In [ ]:
import os, struct
from pathlib import Path

data_path = Path(cfg.data.data_dir)

# Kaggle'daki tüm input dosyalarını listele
print('=== Kaggle Input Dosyaları ===')
for p in sorted(Path('/kaggle/input').rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f'  {str(p):<70}  {size_mb:>8.1f} MB')

print()

# Format tespiti
rec_files  = list(Path('/kaggle/input').rglob('*.rec'))
idx_files  = list(Path('/kaggle/input').rglob('*.idx'))
prop_files = list(Path('/kaggle/input').rglob('property'))

if rec_files:
    print('FORMAT: InsightFace RecordIO tespit edildi')
    print(f'  .rec : {rec_files[0]}')
    print(f'  .idx : {idx_files[0] if idx_files else "YOK"}')
    print(f'  prop : {prop_files[0] if prop_files else "YOK"}')
    DATA_FORMAT = 'recordio'
    REC_PATH    = str(rec_files[0])
    IDX_PATH    = str(idx_files[0]) if idx_files else None
    PROP_PATH   = str(prop_files[0]) if prop_files else None
else:
    id_dirs = [d for d in data_path.iterdir() if d.is_dir()] if data_path.exists() else []
    if id_dirs:
        print('FORMAT: Klasör yapısı (normal)')
        DATA_FORMAT = 'folder'
    else:
        print('UYARI: Veri bulunamadı — DATASET_NAME değişkenini kontrol et')
        DATA_FORMAT = 'none'

In [ ]:
# RecordIO formatını görüntü klasörlerine çıkar
# (Zaten çıkardıysan bu hücreyi atla)

import struct, os
from pathlib import Path
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

EXTRACTED_DIR = '/kaggle/working/ms1mv3_images'

# ── MXNet RecordIO sabitleri ──────────────────────────────────────────────────
# Kayıt yapısı: [magic:4B LE uint32][length:4B LE uint32][IRHeader:24B][jpg:?B]
# IRHeader    : [flag:4B int32][label:4B float32][id:8B uint64][id2:8B uint64]
# length      : IRHeader(24B) + jpg boyutu  (prefix 8B dahil DEĞİL)
_MAGIC       = 0xFEED
_PREFIX_FMT  = '<II'          # magic + length
_PREFIX_SZ   = struct.calcsize(_PREFIX_FMT)   # 8
_IRHDR_FMT   = '<IfQQ'        # flag(int32) + label(float32) + id(uint64) + id2(uint64)
_IRHDR_SZ    = struct.calcsize(_IRHDR_FMT)    # 24

# ── .idx dosyasını oku (binary: her kayıt 16 byte = rec_id(8B) + offset(8B)) ──
def read_idx(idx_path):
    offsets = {}
    # Önce binary dene (InsightFace standart formatı)
    try:
        with open(idx_path, 'rb') as f:
            while True:
                raw = f.read(16)
                if len(raw) < 16:
                    break
                rec_id, offset = struct.unpack('<QQ', raw)
                offsets[int(rec_id)] = int(offset)
        if offsets:
            print(f'IDX: binary format, {len(offsets):,} kayit')
            return offsets
    except Exception as e:
        print(f'IDX binary okuma hatasi: {e}')
    # Text formatı dene (tab-separated)
    offsets = {}
    with open(idx_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                offsets[int(parts[0])] = int(parts[1])
    print(f'IDX: text format, {len(offsets):,} kayit')
    return offsets

# ─────────────────────────────────────────────────────────────────────────────

if DATA_FORMAT != 'recordio':
    print('RecordIO degil, bu hücre atlandı.')

elif os.path.exists(EXTRACTED_DIR) and len(list(Path(EXTRACTED_DIR).iterdir())) > 0:
    n_ids = len(list(Path(EXTRACTED_DIR).iterdir()))
    print(f'Zaten cikarilmis: {EXTRACTED_DIR}  ({n_ids:,} kimlik)')
    cfg.data.data_dir = EXTRACTED_DIR

else:
    # property
    if PROP_PATH and os.path.exists(PROP_PATH):
        with open(PROP_PATH, 'r') as f:
            parts = f.read().strip().split(',')
        print(f'property: {parts}')

    # idx
    offsets = read_idx(IDX_PATH) if IDX_PATH else {}
    if not offsets:
        raise RuntimeError('IDX dosyasi okunamadi veya bos!')

    # İlk birkaç kaydı test et
    print('\nIlk 3 kayıt test ediliyor...')
    errors = 0
    with open(REC_PATH, 'rb') as rf:
        for test_id in list(sorted(offsets.keys()))[1:4]:  # 0 skip (metadata)
            rf.seek(offsets[test_id])
            prefix = rf.read(_PREFIX_SZ)
            magic, length = struct.unpack(_PREFIX_FMT, prefix)
            hdr_raw = rf.read(_IRHDR_SZ)
            flag, label, uid, uid2 = struct.unpack(_IRHDR_FMT, hdr_raw)
            img_sz = length - _IRHDR_SZ
            img_bytes = rf.read(img_sz)
            try:
                img = Image.open(BytesIO(img_bytes))
                print(f'  rec_id={test_id}: magic=0x{magic:04X} label={label:.0f} img={img.size} ok')
            except Exception as e:
                print(f'  rec_id={test_id}: magic=0x{magic:04X} HATA={e}')
                errors += 1

    if errors == 3:
        raise RuntimeError('Hiçbir görüntü decode edilemedi — format hatası!')

    # Çıkarım
    MAX_PER_ID = 30
    MAX_IDS    = 93000

    os.makedirs(EXTRACTED_DIR, exist_ok=True)
    saved_total = 0
    id_counts   = {}

    with open(REC_PATH, 'rb') as rec_f:
        with tqdm(total=min(len(offsets), MAX_IDS * MAX_PER_ID),
                  desc='Goruntu cikariliyor', unit='img') as pbar:
            for rec_id in sorted(offsets.keys()):
                if rec_id == 0:
                    continue

                rec_f.seek(offsets[rec_id])
                prefix = rec_f.read(_PREFIX_SZ)
                if len(prefix) < _PREFIX_SZ:
                    break
                magic, length = struct.unpack(_PREFIX_FMT, prefix)
                if magic != _MAGIC:
                    continue

                hdr_raw = rec_f.read(_IRHDR_SZ)
                flag, label, uid, uid2 = struct.unpack(_IRHDR_FMT, hdr_raw)
                label_id = int(label)

                img_sz = length - _IRHDR_SZ
                if img_sz <= 0:
                    continue
                img_bytes = rec_f.read(img_sz)

                cnt = id_counts.get(label_id, 0)
                if cnt >= MAX_PER_ID:
                    continue
                if MAX_IDS > 0 and label_id >= MAX_IDS:
                    break

                id_dir = os.path.join(EXTRACTED_DIR, f'id_{label_id:06d}')
                os.makedirs(id_dir, exist_ok=True)

                try:
                    img = Image.open(BytesIO(img_bytes))
                    img.save(os.path.join(id_dir, f'{cnt:04d}.jpg'), quality=95)
                    id_counts[label_id] = cnt + 1
                    saved_total += 1
                    pbar.update(1)
                except Exception:
                    continue

    print(f'\nCikarim tamamlandi!')
    print(f'  Kimlik  : {len(id_counts):,}')
    print(f'  Goruntu : {saved_total:,}')
    cfg.data.data_dir = EXTRACTED_DIR

In [ ]:
from pathlib import Path

data_path = Path(cfg.data.data_dir)

if not data_path.exists():
    print(f'HATA: {data_path} bulunamadi!')
    print('extract_recordio hucresini calistirdin mi?')
else:
    id_dirs = [d for d in data_path.iterdir() if d.is_dir()]
    if not id_dirs:
        print(f'HATA: {data_path} bos — cikarim basarisiz olmus olabilir.')
        print('extract_recordio hucresini tekrar calistir.')
    else:
        sample = id_dirs[:200]
        total  = sum(len(list(d.glob('*.jpg'))) for d in sample)
        avg    = total // len(sample)
        print(f'Veri klasoru  : {data_path}')
        print(f'Kimlik sayisi : {len(id_dirs):,}')
        print(f'Ort. goruntu  : ~{avg} per kimlik')
        print(f'Toplam tahmin : ~{len(id_dirs) * avg:,}')
        print('Hazir!')

## 4b. IResNet50-ArcFace Ağırlıklarını İndir (Identity Supervisor)

Bu model **yalnızca eğitimde** kimlik denetleyicisi olarak kullanılır. Telefona yüklenmez.

InsightFace'in resmi MS1MV3-ArcFace R50 backbone dosyasını indir.
> Dosya ~87 MB, bir kez indirilir.

In [ ]:
import os, urllib.request

ARCFACE_PATH = '/kaggle/working/arcface_r50.pth'

if os.path.exists(ARCFACE_PATH):
    size_mb = os.path.getsize(ARCFACE_PATH) / 1e6
    print(f'IResNet50 ağırlıkları mevcut: {size_mb:.1f} MB')
else:
    # InsightFace arcface_torch MS1MV3-R50 backbone
    # Kaynak: https://github.com/deepinsight/insightface/tree/master/recognition/arcface_torch
    URL = 'https://github.com/deepinsight/insightface/releases/download/v0.7/ms1mv3_arcface_r50_fp16.zip'
    ZIP = '/kaggle/working/arcface_r50.zip'

    print('IResNet50 ağırlıkları indiriliyor...')
    try:
        urllib.request.urlretrieve(URL, ZIP)
        import zipfile
        with zipfile.ZipFile(ZIP, 'r') as zf:
            print('ZIP içeriği:', zf.namelist())
            # backbone.pth veya benzer isim
            pth_files = [n for n in zf.namelist() if n.endswith('.pth')]
            if pth_files:
                with zf.open(pth_files[0]) as src, open(ARCFACE_PATH, 'wb') as dst:
                    dst.write(src.read())
                print(f'Kaydedildi: {ARCFACE_PATH}')
            else:
                print('UYARI: ZIP içinde .pth bulunamadı, dosyalar:', zf.namelist())
                ARCFACE_PATH = None
        os.remove(ZIP)
    except Exception as e:
        print(f'İndirme başarısız: {e}')
        print('FaceNet identity loss ile devam edilecek (yeterli ama optimal değil)')
        ARCFACE_PATH = None

print(f'ARCFACE_PATH = {ARCFACE_PATH}')

## 5. Ön İşleme

In [ ]:
# Daha önce işlediysen ve /kaggle/working/processed varsa atla
manifest_path = '/kaggle/working/processed/manifest.txt'

if os.path.exists(manifest_path):
    print(f'İşlenmiş veri zaten mevcut: {manifest_path}')
    with open(manifest_path) as f:
        n = sum(1 for l in f if l.strip())
    print(f'Örnek sayısı: {n:,}')
else:
    from src.data.preprocess import preprocess_dataset
    manifest_path = preprocess_dataset(
        raw_dir     = cfg.data.data_dir,
        out_dir     = cfg.data.processed_dir,
        output_size = cfg.data.image_size,
        max_per_id  = 100,
    )

In [ ]:
from src.data.dataset import build_dataloaders
import torch

train_loader, val_loader = build_dataloaders(
    manifest_path = manifest_path,
    image_size    = cfg.data.image_size,
    batch_size    = cfg.train.batch_size,
    val_split     = cfg.data.val_split,
    num_workers   = cfg.data.num_workers,
)
print(f'Train: {len(train_loader.dataset):,}  |  Val: {len(val_loader.dataset):,}')

emb, img = next(iter(train_loader))
print(f'Embedding: {emb.shape}  |  Görüntü: {img.shape}  |  Aralık: [{img.min():.2f}, {img.max():.2f}]')

## 6. Eğitim

In [ ]:
from src.train import train

# Önceki session'dan checkpoint varsa devam et
resume_path = '/kaggle/working/checkpoints/best_model.pt'
if not os.path.exists(resume_path):
    resume_path = None

if resume_path:
    print(f'Checkpoint bulundu, devam ediliyor: {resume_path}')
else:
    print('Sıfırdan eğitim başlıyor...')

best_ckpt = train(
    cfg           = cfg,
    manifest_path = manifest_path,
    resume_from   = resume_path,
)
print(f'Eğitim tamamlandı: {best_ckpt}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/logs

## 7. Sonuçları Görselleştir

In [ ]:
import torch, matplotlib.pyplot as plt
from src.models.decoder import FaceDecoder
from src.data.dataset import denormalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = FaceDecoder(
    embedding_dim    = cfg.model.embedding_dim,
    initial_spatial  = cfg.model.initial_spatial,
    initial_channels = cfg.model.initial_channels,
    decoder_channels = cfg.model.decoder_channels,
).to(device)

state = torch.load(best_ckpt, map_location=device)
model.load_state_dict(state['model'])
model.eval()

val_embs, val_imgs = next(iter(val_loader))
with torch.no_grad():
    generated = model(val_embs[:8].to(device)).cpu()

real_np = denormalize(val_imgs[:8]).permute(0,2,3,1).numpy()
gen_np  = denormalize(generated).permute(0,2,3,1).numpy()

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i in range(8):
    axes[0,i].imshow(real_np[i].clip(0,1)); axes[0,i].set_title('Gerçek', fontsize=8); axes[0,i].axis('off')
    axes[1,i].imshow(gen_np[i].clip(0,1));  axes[1,i].set_title('Üretilen', fontsize=8); axes[1,i].axis('off')
plt.suptitle('Gerçek vs Üretilen', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/results_sample.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. TFLite Export

In [ ]:
from src.export_tflite import export_model

exported = export_model(
    checkpoint_path = best_ckpt,
    export_dir      = cfg.export.export_dir,
    cfg             = cfg,
)

print('\nExport tamamlandı:')
for fmt, path in exported.items():
    size_mb = os.path.getsize(path) / 1e6
    print(f'  {fmt:12s}: {size_mb:.2f} MB  →  {path}')

## 9. Checkpoint'i Kalıcı Kaydet

Kaggle session kapandığında `/kaggle/working/` silinir.  
Checkpoint'i **Kaggle Dataset** olarak kaydetmek için:

In [ ]:
# Önemli dosyaları tek zip'e topla
import zipfile, shutil

zip_path = '/kaggle/working/face_recon_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Best model checkpoint
    if os.path.exists('/kaggle/working/checkpoints/best_model.pt'):
        zf.write('/kaggle/working/checkpoints/best_model.pt', 'best_model.pt')
    # TFLite dosyaları
    for fmt, path in exported.items():
        if os.path.exists(path):
            zf.write(path, os.path.basename(path))
    # Örnek görüntüler
    if os.path.exists('/kaggle/working/results_sample.png'):
        zf.write('/kaggle/working/results_sample.png', 'results_sample.png')

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zip oluşturuldu: {zip_path}  ({size_mb:.1f} MB)')
print()
print('Kalıcı kaydetmek için:')
print('  1. Sağ panelde Output sekmesine tıkla')
print('  2. face_recon_results.zip yanındaki üç noktaya tıkla')
print('  3. "Save to Dataset" seç → yeni dataset oluştur')
print('  4. Sonraki session\'da bu dataseti notebook\'a ekle')

In [ ]:
# Sonraki session'da checkpoint'i geri yüklemek için:
# 1. Kaggle dataset'ini notebook'a ekle
# 2. Aşağıdaki kodu çalıştır:

# import shutil, zipfile
# with zipfile.ZipFile('/kaggle/input/DATASET_ADIN/face_recon_results.zip') as zf:
#     zf.extract('best_model.pt', '/kaggle/working/checkpoints/')
# print('Checkpoint geri yüklendi.')